In [1]:
using TypedTables
using BenchmarkTools
using PrettyPrint

In [2]:
using CovidSim_ilm

In [3]:
n = 100_000
locdat = CovidSim_ilm.pop_data(n, age_dist=CovidSim_ilm.age_dist)

Table with 18 columns and 100000 rows:
      pid  status     agegrp   cond     duration  variant  recovday  deadday  ⋯
    ┌─────────────────────────────────────────────────────────────────────────
 1  │ 1    unexposed  age0_19  notsick  0        default  0         0        ⋯
 2  │ 2    unexposed  age0_19  notsick  0        default  0         0        ⋯
 3  │ 3    unexposed  age0_19  notsick  0        default  0         0        ⋯
 4  │ 4    unexposed  age0_19  notsick  0        default  0         0        ⋯
 5  │ 5    unexposed  age0_19  notsick  0        default  0         0        ⋯
 6  │ 6    unexposed  age0_19  notsick  0        default  0         0        ⋯
 7  │ 7    unexposed  age0_19  notsick  0        default  0         0        ⋯
 8  │ 8    unexposed  age0_19  notsick  0        default  0         0        ⋯
 9  │ 9    unexposed  age0_19  notsick  0        default  0         0        ⋯
 10 │ 10   unexposed  age0_19  notsick  0        default  0         0        ⋯
 11 │ 11   u

In [5]:
vaxset = CovidSim_ilm.build_vaxset("vaccines.yml")

Dict{Any, Any} with 3 entries:
  :Moderna => Dict{Symbol, Any}(:transition=>Dict{agegrp, Dict{Int64, Dict{Symb…
  :Pfizer  => Dict{Symbol, Any}(:transition=>Dict{agegrp, Dict{Int64, Dict{Symb…
  :JnJ     => Dict{Symbol, Any}(:transition=>Dict{agegrp, Dict{Int64, Dict{Symb…

In [6]:
vaxset[:Pfizer]

Dict{Symbol, Any} with 2 entries:
  :transition => Dict{agegrp, Dict{Int64, Dict{Symbol, Any}}}(age40_59=>Dict(5=…
  :params     => Vaccineparams(:Pfizer, 2, 24, 360, [0.0, 0.1, 0.2, 0.15, 0.15,…

In [7]:
vaxset[:Pfizer][:transition]

Dict{agegrp, Dict{Int64, Dict{Symbol, Any}}} with 5 entries:
  age40_59 => Dict(5=>Dict(:transition=>[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; …
  age60_79 => Dict(5=>Dict(:transition=>[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; …
  age0_19  => Dict(5=>Dict(:transition=>[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; …
  age20_39 => Dict(5=>Dict(:transition=>[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; …
  age80_up => Dict(5=>Dict(:transition=>[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; …

In [8]:
vaccines = collect(keys(vaxset))

3-element Vector{Any}:
 :Moderna
 :Pfizer
 :JnJ

In [9]:
vsdict = CovidSim_ilm.build_vaxschedset()

Dict{Symbol, Vaxsched} with 1 entry:
  :us1 => Vaxsched(Dict{Symbol, Vaxinclude}(:Moderna=>Vaxinclude(0.3, 25000, 0.…

In [11]:
vsdict[:us1].vaxesincluded

Dict{Symbol, Vaxinclude} with 3 entries:
  :Moderna => Vaxinclude(0.3, 25000, 0.85)
  :Pfizer  => Vaxinclude(0.55, 40000, 0.85)
  :JnJ     => Vaxinclude(0.15, 12500, 0.0)

In [12]:
vsdict[:us1].shotmode

:all

In [13]:
fieldnames(typeof(vsdict[:us1]))

(:vaxesincluded, :dayrange, :targetpct, :filterfunc, :shotmode, :pattern, :pctfunc)

In [14]:
day_ctr[:day] = 500

500

In [15]:
cidx = collect(1:n);

In [16]:
CovidSim_ilm.vaccinate!(locdat, vsdict, cidx, vaxset)

In [17]:
count(locdat.vaxstatus .== :first)

295

In [18]:
count(locdat.vaxstatus .== :full)

0

In [20]:
locdat.vaxrcvd[locdat.vaxstatus .!= :none]

295-element Vector{Vector{Symbol}}:
 [:Pfizer]
 [:Moderna]
 [:JnJ]
 [:Pfizer]
 [:Pfizer]
 [:JnJ]
 [:JnJ]
 [:JnJ]
 [:Pfizer]
 [:Moderna]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 ⋮
 [:Pfizer]
 [:Pfizer]
 [:JnJ]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:JnJ]
 [:Pfizer]
 [:JnJ]
 [:Pfizer]
 [:Pfizer]

In [22]:
locdat.vaxrcvd[in.(:Pfizer, locdat.vaxrcvd)]

154-element Vector{Vector{Symbol}}:
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 ⋮
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]
 [:Pfizer]

In [23]:
locdat.vaxrcvd[in.(:Moderna, locdat.vaxrcvd)]

88-element Vector{Vector{Symbol}}:
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 ⋮
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]
 [:Moderna]

In [30]:
@time @Select(vaxstatus, vaxday, vaxrcvd)(locdat)[in.(:Moderna, locdat.vaxrcvd)]

  0.000325 seconds (17 allocations: 19.422 KiB)


Table with 3 columns and 88 rows:
      vaxstatus  vaxday  vaxrcvd
    ┌──────────────────────────────
 1  │ first      [500]   [:Moderna]
 2  │ first      [500]   [:Moderna]
 3  │ first      [500]   [:Moderna]
 4  │ first      [500]   [:Moderna]
 5  │ first      [500]   [:Moderna]
 6  │ first      [500]   [:Moderna]
 7  │ first      [500]   [:Moderna]
 8  │ first      [500]   [:Moderna]
 9  │ first      [500]   [:Moderna]
 10 │ first      [500]   [:Moderna]
 11 │ first      [500]   [:Moderna]
 12 │ first      [500]   [:Moderna]
 13 │ first      [500]   [:Moderna]
 14 │ first      [500]   [:Moderna]
 15 │ first      [500]   [:Moderna]
 16 │ first      [500]   [:Moderna]
 17 │ first      [500]   [:Moderna]
 18 │ first      [500]   [:Moderna]
 19 │ first      [500]   [:Moderna]
 20 │ first      [500]   [:Moderna]
 21 │ first      [500]   [:Moderna]
 22 │ first      [500]   [:Moderna]
 23 │ first      [500]   [:Moderna]
 ⋮  │     ⋮        ⋮         ⋮

## Some performance and method experiments

In [3]:
for i in eachindex(locdat.status)
    locdat.status[i] = rand([dead, unexposed, infectious, recovered])
    locdat.vaxstatus[i] = rand([:none, :full])
end

In [5]:
@btime targetidx =  findall(($locdat.status .!= dead) .& ($locdat.vaxstatus .!= :full) .& 
    ($locdat.agegrp .!= age0_19));

  210.524 μs (5 allocations: 235.92 KiB)


In [6]:
pf = Vaccineparams(name=:Pfizer,reqdshots=2,delay2ndshot=24,halflife=360,
sendrisk=Float64[], recvrisk=Float64[],recvrisk_reduction=Dict(:base=>.94))

Vaccineparams(:Pfizer, 2, 24, 360, Float64[], Float64[], Dict(:base => 0.94))

In [11]:
vs = Vaxsched(
    vaxinclude=Dict(:Pfizer=>CovidSim_ilm.Vaxinclude(mix=.6, doses=30_000, pct2ndshot=.9),
        :Moderna=>CovidSim_ilm.Vaxinclude(mix=0.4, doses=12_000, pct2ndshot=.9)),
    dayrange=35:100, 
    targetpct=.70,
    #filterfunc=CovidSim_ilm.basevaxfilterfunc,
    pattern=[0.0, .02, .05, .10, .15, .19, .21, .16, .08, .03, .01]
    )


Vaxsched(Dict{Symbol, Vaxinclude}(:Moderna => Vaxinclude(0.4, 12000, 0.9), :Pfizer => Vaxinclude(0.6, 30000, 0.9)), 35:100, 0.7, CovidSim_ilm.basevaxfilterfunc, :all, [0.0, 0.02, 0.05, 0.1, 0.15, 0.19, 0.21, 0.16, 0.08, 0.03, 0.01], CovidSim_ilm.var"#vaxpctperday#128"{Float64, Vector{Float64}, Int64, Interpolations.Extrapolation{Float64, 1, Interpolations.ScaledInterpolation{Float64, 1, Interpolations.BSplineInterpolation{Float64, 1, Vector{Float64}, Interpolations.BSpline{Interpolations.Linear{Interpolations.Throw{Interpolations.OnGrid}}}, Tuple{Base.OneTo{Int64}}}, Interpolations.BSpline{Interpolations.Linear{Interpolations.Throw{Interpolations.OnGrid}}}, Tuple{UnitRange{Int64}}}, Interpolations.BSpline{Interpolations.Linear{Interpolations.Throw{Interpolations.OnGrid}}}, Interpolations.Throw{Nothing}}, Int64}(0.7, [0.0, 0.02, 0.05, 0.1, 0.15, 0.19, 0.21, 0.16, 0.08, 0.03, 0.01], 35, 11-element extrapolate(scale(interpolate(::Vector{Float64}, BSpline(Linear())), (0:10,)), Throw()) wit

In [2]:
limit = 10_000
x = collect(1:limit)
newidx = Vector{Int}(undef, length(x))
oddidx = findall(isodd.(x))


5000-element Vector{Int64}:
    1
    3
    5
    7
    9
   11
   13
   15
   17
   19
   21
   23
   25
    ⋮
 9977
 9979
 9981
 9983
 9985
 9987
 9989
 9991
 9993
 9995
 9997
 9999

In [3]:
flipper(x) = flipsign(x, -1)
ranger(x) = 500 < x < 1_000
function foobar!(x, firstidx, critfunc, applyfunc)
    for i in firstidx
        critfunc(x[i]) && (applyfunc(x[i])) 
    end
end

foobar! (generic function with 1 method)

In [4]:
@btime foobar!($x, oddidx, ranger, flipper);

  2.261 μs (0 allocations: 0 bytes)


In [5]:
x

10000-element Vector{Int64}:
     1
     2
     3
     4
     5
     6
     7
     8
     9
    10
    11
    12
    13
     ⋮
  9989
  9990
  9991
  9992
  9993
  9994
  9995
  9996
  9997
  9998
  9999
 10000

In [7]:
@time foo = [i for i in oddidx if ranger(x[i])];

  0.056306 seconds (73.30 k allocations: 3.719 MiB, 103.87% compilation time)


In [8]:
orderdict = Dict(:one => 1, :two => 2, :three => 3)

Dict{Symbol, Int64} with 3 entries:
  :three => 3
  :two   => 2
  :one   => 1

In [9]:
ks = collect(keys(orderdict))

3-element Vector{Symbol}:
 :three
 :two
 :one

In [10]:
vs = collect(values(orderdict))

3-element Vector{Int64}:
 3
 2
 1

In [11]:
ks[1]

:three

In [12]:
last(ks)

:one

In [13]:
nums = [1,2,3]
this = 3
#@benchmark union(this, nums) setup=(this=[2], nums = [1,2,3])
union(this, nums)



3-element Vector{Int64}:
 3
 1
 2

In [14]:
newidx

10000-element Vector{Int64}:
 140372416135168
 140375661764632
     17179869184
               0
 140375661764656
     17179869184
 140375661764672
     17179869184
 140375661780662
      4294967296
 140372416135168
 140375661764712
     17179869184
               ⋮
               0
               0
               0
               0
               0
               0
               0
               0
               0
               0
               0
               0

In [ ]:
y(x) = 